# Topic 34 — LSTM
### ⭐ Very important for your NLP path. Theory → gates from scratch → nn.LSTM → full text classifier.

**LSTM (Long Short-Term Memory)** fixes the vanishing gradient problem from Topic 33 by adding a
separate **cell state** (a "conveyor belt" of long-term memory) alongside the hidden state, plus
three **gates** that learn to control what information flows through:

- **Forget gate**: decides what to THROW AWAY from the cell state.
- **Input gate**: decides what NEW information to ADD to the cell state.
- **Output gate**: decides what part of the cell state to OUTPUT as the hidden state.

Each gate is its own small neural network layer (sigmoid output, 0-1 range) that multiplies
("gates") a piece of information — near 0 blocks it, near 1 lets it through.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. LSTM gates — from scratch

At each timestep, given input `x_t`, previous hidden state `h_{t-1}`, and previous cell state
`C_{t-1}`:

```text
forget_gate = sigmoid(W_f @ [h_{t-1}, x_t] + b_f)
input_gate  = sigmoid(W_i @ [h_{t-1}, x_t] + b_i)
candidate   = tanh(W_c @ [h_{t-1}, x_t] + b_c)        # new candidate values to possibly add
C_t = forget_gate * C_{t-1} + input_gate * candidate   # update cell state
output_gate = sigmoid(W_o @ [h_{t-1}, x_t] + b_o)
h_t = output_gate * tanh(C_t)                          # update hidden state
```

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

def lstm_cell_step(x_t, h_prev, C_prev, weights):
    combined = np.concatenate([h_prev, x_t])   # [h_{t-1}, x_t] stacked together

    forget_gate = sigmoid(weights["Wf"] @ combined + weights["bf"])
    input_gate = sigmoid(weights["Wi"] @ combined + weights["bi"])
    candidate = np.tanh(weights["Wc"] @ combined + weights["bc"])
    C_t = forget_gate * C_prev + input_gate * candidate

    output_gate = sigmoid(weights["Wo"] @ combined + weights["bo"])
    h_t = output_gate * np.tanh(C_t)

    return h_t, C_t, {"forget": forget_gate, "input": input_gate, "output": output_gate}

hidden_dim, input_dim = 4, 3
rng = np.random.default_rng(0)
combined_dim = hidden_dim + input_dim
weights = {
    "Wf": rng.normal(0, 0.3, (hidden_dim, combined_dim)), "bf": np.zeros(hidden_dim),
    "Wi": rng.normal(0, 0.3, (hidden_dim, combined_dim)), "bi": np.zeros(hidden_dim),
    "Wc": rng.normal(0, 0.3, (hidden_dim, combined_dim)), "bc": np.zeros(hidden_dim),
    "Wo": rng.normal(0, 0.3, (hidden_dim, combined_dim)), "bo": np.zeros(hidden_dim),
}

h, C = np.zeros(hidden_dim), np.zeros(hidden_dim)
X_seq = rng.normal(0, 1, size=(5, input_dim))

for t in range(5):
    h, C, gates = lstm_cell_step(X_seq[t], h, C, weights)
    print(f"t={t}: forget_gate={np.round(gates['forget'],2)}  input_gate={np.round(gates['input'],2)}")
# The gates decide, at every step, how much old memory to keep vs how much new info to add --
# this is what lets LSTMs preserve information across MANY more timesteps than plain RNNs.

## 2. `nn.LSTM` in PyTorch

Same interface pattern as `nn.RNN` (Topic 33), but returns BOTH the hidden state AND cell state.

In [ ]:
lstm_layer = nn.LSTM(input_size=3, hidden_size=4, batch_first=True)

X_batch = torch.tensor(X_seq, dtype=torch.float32).unsqueeze(0)
output, (h_n, c_n) = lstm_layer(X_batch)

print("output shape (hidden state at every timestep):", output.shape)
print("h_n shape (final hidden state):", h_n.shape)
print("c_n shape (final cell state -- LSTM's extra 'long-term memory'):", c_n.shape)

## 3. Full text classification pipeline: Text -> Tokenization -> Embedding -> LSTM -> Linear

This is the architecture named explicitly in the roadmap. We build every piece:
1. Tokenize text into words (Topic 21).
2. Build a vocabulary and convert words to integer indices.
3. `nn.Embedding` turns each index into a learnable dense vector (Topic 26's idea, but now the
   embeddings are learned END-TO-END for this specific task, not pretrained separately).
4. `nn.LSTM` processes the sequence of embeddings.
5. A final `nn.Linear` classifies based on the LSTM's last hidden state.

In [ ]:
bullying_examples = [
    "you are stupid and worthless", "i hate you so much", "get lost loser",
    "nobody wants you here", "you should just disappear", "you are pathetic",
    "everyone thinks you're an idiot", "just go away nobody likes you",
    "you're so ugly and useless", "why do you even exist",
    "you deserve to be alone", "stop talking you sound stupid",
]
not_bullying_examples = [
    "great job today team", "have a wonderful day", "nice work everyone",
    "thanks for your help", "well done on the project", "excellent effort today",
    "looking forward to the weekend", "congratulations on your achievement",
    "the weather is nice today", "let's grab coffee sometime",
    "i really appreciate your feedback", "the meeting went smoothly",
]
texts = bullying_examples + not_bullying_examples
labels = [1]*len(bullying_examples) + [0]*len(not_bullying_examples)

# --- Step 1 & 2: tokenize and build a vocabulary ---
def tokenize(text):
    return text.lower().split()

all_words = set(word for text in texts for word in tokenize(text))
vocab = {"<PAD>": 0, "<UNK>": 1}   # PAD for padding shorter sequences, UNK for unknown words
for word in sorted(all_words):
    vocab[word] = len(vocab)

print("vocabulary size:", len(vocab))

def text_to_indices(text, vocab, max_len=8):
    tokens = tokenize(text)
    indices = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
    indices = indices[:max_len]                                    # truncate if too long
    indices += [vocab["<PAD>"]] * (max_len - len(indices))          # pad if too short
    return indices

print("\nexample:", texts[0], "->", text_to_indices(texts[0], vocab))

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=8):
        self.X = [torch.tensor(text_to_indices(t, vocab, max_len)) for t in texts]
        self.y = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

X_train_txt, X_test_txt, y_train_lbl, y_test_lbl = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

train_dataset = TextDataset(X_train_txt, y_train_lbl, vocab)
test_dataset = TextDataset(X_test_txt, y_test_lbl, vocab)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

## 4. The model architecture

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32, n_classes=2, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        embedded = self.embedding(x)          # (batch, seq_len) -> (batch, seq_len, embed_dim)
        _, (h_n, c_n) = self.lstm(embedded)    # process the whole sequence
        h_final = h_n.squeeze(0)                # (batch, hidden_dim) -- summary of the whole sentence
        return self.fc(h_final)                 # classify from that summary

model = LSTMClassifier(vocab_size=len(vocab)).to(device)
print(model)
print("\ntotal parameters:", sum(p.numel() for p in model.parameters()))

## 5. Training loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def accuracy(loader, model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            correct += (model(Xb).argmax(1) == yb).sum().item()
            total += yb.size(0)
    return correct / total

losses, test_accs = [], []
for epoch in range(60):
    model.train()
    epoch_losses = []
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    losses.append(np.mean(epoch_losses))
    test_accs.append(accuracy(test_loader, model))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(losses); axes[0].set_title("training loss"); axes[0].set_xlabel("epoch")
axes[1].plot(test_accs); axes[1].set_title("test accuracy"); axes[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

print("final test accuracy:", test_accs[-1])

## 6. Try it on new sentences

In [ ]:
def predict_sentence(sentence, model, vocab):
    model.eval()
    indices = torch.tensor([text_to_indices(sentence, vocab)]).to(device)
    with torch.no_grad():
        logits = model(indices)
        probs = torch.softmax(logits, dim=1)
    pred = probs.argmax(1).item()
    return ("bullying" if pred == 1 else "not bullying"), probs[0].cpu().numpy()

for sentence in ["you are worthless and stupid", "have an amazing day everyone", "you are such a loser"]:
    label, probs = predict_sentence(sentence, model, vocab)
    print(f"'{sentence}' -> {label}  (probs: not_bullying={probs[0]:.2f}, bullying={probs[1]:.2f})")

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Add 10 more of your own examples to bullying_examples/not_bullying_examples, rebuild the
#    vocabulary and datasets, and retrain -- watch how accuracy and predictions change.
# 2. Change hidden_dim from 32 to 64 and embed_dim from 16 to 32 -- does it help on this tiny dataset,
#    or does it overfit even faster (reconnect to Topic 31)?
# 3. Replace nn.LSTM with nn.RNN (Topic 33) in LSTMClassifier, keep everything else the same,
#    and compare final test accuracy -- with such short sentences, does the difference show up much?
#    (LSTM's advantage grows with LONGER sequences -- this toy example may not show a big gap.)
# 4. Once you load your real cyberbullying dataset, increase max_len in text_to_indices to fit
#    your actual comment lengths (check df["text"].str.split().str.len().describe() first, Topic 2).

---
### Next up: **Topic 35 — GRU** (a lighter alternative to LSTM).

Say "next" when you're ready.